In [1]:
import datetime
from preproc import *
import pandas as pd
import numpy as np
from psaw import PushshiftAPI

In [2]:
pd.set_option("max_rows", 50)
pd.options.plotting.backend = "plotly" # only fucking working 

In [3]:
pushshift_api = PushshiftAPI()
subreddit = "wallstreetbets"
limit = 100

from_date = {"YEAR" : 2021, "MONTH" : 1, "DAY" : 1, "HOUR" : 20, "MINUTE" : 0, "SECOND" : 0}
to_date   = {"YEAR" : 2021, "MONTH" : 1, "DAY" : 2, "HOUR" : 0, "MINUTE" : 0, "SECOND" : 0}

# Test
## Testing get_submissions method

In [13]:
%load_ext autoreload
%autoreload 2

df = get_submissions(posted_after_date, posted_before_date, subreddit, limit)
df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,id,author,created_utc,datetime,domain,url,title,score,selftext,num_comments,num_crossposts,full_link,link_flair_text
99,lags57,[deleted],1612220193,2021-02-01 23:56:33,self.wallstreetbets,https://www.reddit.com/r/wallstreetbets/commen...,I'm finally in.,1,NaN,2,0,https://www.reddit.com/r/wallstreetbets/commen...,YOLO
98,lags5j,Forsaken_Double9048,1612220193,2021-02-01 23:56:33,self.wallstreetbets,https://www.reddit.com/r/wallstreetbets/commen...,Spce what do I do,1,[removed],0,0,https://www.reddit.com/r/wallstreetbets/commen...,Discussion
97,lags5z,Ileflo,1612220194,2021-02-01 23:56:34,i.redd.it,https://i.redd.it/2v0p2zuu4ye61.png,GME Suffering in the aftermarket! All we can d...,1,,0,0,https://www.reddit.com/r/wallstreetbets/commen...,Chart
96,lags72,Noonish1234,1612220196,2021-02-01 23:56:36,i.redd.it,https://i.redd.it/fniqk25v4ye61.jpg,End of day update. I’m not selling!,63,,1,0,https://www.reddit.com/r/wallstreetbets/commen...,Loss
95,lags7p,[deleted],1612220196,2021-02-01 23:56:36,self.wallstreetbets,https://www.reddit.com/r/wallstreetbets/commen...,Glad GME is dropping...,2,[deleted],8,0,https://www.reddit.com/r/wallstreetbets/commen...,Discussion


# Test
## Parallelizing ingestion pipeline 
    TOTAL FAILURE F*****G WINDOWS

In [6]:
from joblib import Parallel, delayed
from math import sqrt

In [11]:
Parallel(n_jobs=2)(delayed(sqrt)(i ** 2) for i in range(10))

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.


In [21]:
import multiprocessing

print("Number of cpu : ", multiprocessing.cpu_count())

Number of cpu :  8


In [30]:
from multiprocessing import Process

def f(name):
    print('hello', name)

p = Process(target=f, args=('bob',))
p.start()
p.join()

In [40]:
posted_after_date_1, posted_before_date_1 = {"YEAR" : 2021, "MONTH" : 1, "DAY" : 1, "HOUR" : 23, "MINUTE" : 0, "SECOND" : 0}, {"YEAR" : 2021, "MONTH" : 1, "DAY" : 2, "HOUR" : 0, "MINUTE" : 0, "SECOND" : 0}
posted_after_date_2, posted_before_date_2 = {"YEAR" : 2021, "MONTH" : 1, "DAY" : 1, "HOUR" : 22, "MINUTE" : 0, "SECOND" : 0}, {"YEAR" : 2021, "MONTH" : 1, "DAY" : 1, "HOUR" : 23, "MINUTE" : 0, "SECOND" : 0}
subreddit = "wallstreetbets"
limit = 1000

preproc = [
        (posted_after_date_1, posted_before_date_1, subreddit, limit),
        (posted_after_date_2, posted_before_date_2, subreddit, limit)
    ]

"""
Working only with = n_jobs=1
"""
res = Parallel(n_jobs=1, verbose=100)(
    delayed(parse_comments)(*p_list) # p_list[0], p_list[1], p_list[2], p_list[3] 
    for p_list in preproc
)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
utc_lower_bound 1609538400 utc_upper_bound 1609542000 formatted utc_lower_bound 01/01/2021-23/00/00 formatted utc_upper_bound 02/01/2021-00/00/00
first_utc_in_time 1609538548 last_utc_in_time 1609541754 formatted first_utc_in_time 01/01/2021-23/02/28 formatted last_utc_in_time 01/01/2021-23/55/54
utc_lower_bound 1609538400 utc_upper_bound 1609538548 formatted utc_lower_bound 01/01/2021-23/00/00 formatted utc_upper_bound 01/01/2021-23/02/28
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    9.9s remaining:    0.0s
utc_lower_bound 1609534800 utc_upper_bound 1609538400 formatted utc_lower_bound 01/01/2021-22/00/00 formatted utc_upper_bound 01/01/2021-23/00/00


# PIPELINE 
## submissions ingestion

In [6]:
%%time
%load_ext autoreload
%autoreload 2

pushshift_api = PushshiftAPI()
subreddit = "wallstreetbets"
limit = 1000

from_date = {"YEAR" : 2021, "MONTH" : 1, "DAY" : 1, "HOUR" : 22, "MINUTE" : 0, "SECOND" : 0}
to_date   = {"YEAR" : 2021, "MONTH" : 1, "DAY" : 2, "HOUR" : 0, "MINUTE" : 0, "SECOND" : 0}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Wall time: 715 ms


In [7]:
%%time
submissions = parse_comments(from_date, to_date, subreddit, limit)

submissions["COMMENTS_NUM"] = submissions[submissions.COMMENTS_DATA.notnull()].swifter.apply(lambda x: len(x.COMMENTS_DATA), axis=1)

comments_tmp = pd.DataFrame()
for d_comm in submissions[submissions.COMMENTS_DATA.notnull()].COMMENTS_DATA:
    comments_tmp = pd.concat([comments_tmp, d_comm])

submissions["HAS_COMMENTS"] = submissions.COMMENTS_DATA.notnull()
submissions.drop(columns=["COMMENTS_DATA"], inplace=True)

comments_tmp[["PARENT_TYPE", "PARENT_ID"]] = comments_tmp.apply(lambda x: x.parent_id.split("_"), axis=1, result_type='expand')
comments_tmp.sort_values("timestamp", inplace=True)
comments_tmp.set_index("id", inplace=True)
comments_tmp.shape

utc_lower_bound 1609534800 utc_upper_bound 1609542000 formatted utc_lower_bound 01/01/2021-22/00/00 formatted utc_upper_bound 02/01/2021-00/00/00


KeyboardInterrupt: 

# Analysis
## submissions dataframe
## comments_tmp dataframe

In [ ]:
submissions["datetime"].iloc[0], submissions["datetime"].iloc[-1], submissions["datetime"].min(), submissions["datetime"].max(), submissions["datetime"].max() - submissions["datetime"].min()

In [262]:
submissions.shape, comments_tmp.shape

((28, 15), (228, 14))

In [263]:
comments_tmp[comments_tmp.parent_id == comments_tmp.submission_id].PARENT_TYPE.value_counts(), comments_tmp[comments_tmp.parent_id != comments_tmp.submission_id].PARENT_TYPE.value_counts()

(t3    119
 Name: PARENT_TYPE, dtype: int64,
 t1    109
 Name: PARENT_TYPE, dtype: int64)

In [271]:
submissions.link_flair_text.value_counts()

Discussion    10
Meme           8
YOLO           3
Chart          3
Gain           3
DD             1
Name: link_flair_text, dtype: int64

In [273]:
submissions[["COMMENTS_NUM", "num_comments"]].sum()

COMMENTS_NUM    228.0
num_comments    267.0
dtype: float64

In [367]:
submissions.sort_values("num_comments", ascending=False)[["title", "link_flair_text", "COMMENTS_NUM", "url", "id", "datetime", "num_comments", "score"]].head(2)

,title,link_flair_text,COMMENTS_NUM,url,id,datetime,num_comments,score
21,Dear student retards...,Discussion,102.0,https://www.reddit.com/r/wallstreetbets/commen...,kokdo8,2021-01-01 23:15:05,117,1
24,Ryan Cohen-GME confirmation bias,Chart,54.0,https://i.redd.it/xv2ie5t8ns861.jpg,kok7my,2021-01-01 23:05:48,57,1


# CHECKPOINT
## Export submissions.parquet
## Export comments.parquet

In [ ]:
submissions.to_parquet("submissions.parquet", engine="fastparquet")
submissions_test_import = pd.read_parquet("submissions.parquet", engine="fastparquet")
submissions_test_import.head()

In [ ]:
comments_tmp.reset_index().drop(columns=["replies", "submission", "subreddit"]).to_parquet("comments_tmp.parquet", engine="fastparquet")
comments_tmp_test_import = pd.read_parquet("comments_tmp.parquet", engine="fastparquet")
comments_tmp_test_import.head()

In [ ]:
comments_tmp.head()

# Analysis
## Getting moreComments (commentType object) from replies column

In [93]:
d_comments.replies

<bound method Series.append of 0            ()
1            ()
1     (ghrlhzv)
0            ()
2     (ghrkqng)
        ...    
0            ()
65           ()
66           ()
3            ()
9            ()
Name: replies, Length: 228, dtype: object>

In [134]:
d_comments[d_comments.submission == "kokdo8"].shape[0] == df_global[df_global.id == "kokdo8"].COMMENTS_NUM

21    True
Name: COMMENTS_NUM, dtype: bool

In [166]:
d_comments[d_comments.PARENT_ID == "kokafr"].replies.iloc[1].__getitem__(0)
d_comments[d_comments.PARENT_ID == "kokafr"].replies.iloc[1].__getitem__(0).id
d_comments[d_comments.PARENT_ID == "kokafr"].replies.iloc[1].__getitem__(0).body
d_comments[d_comments.PARENT_ID == "kokafr"].replies.iloc[1].__getitem__(0).created_utc
d_comments[d_comments.PARENT_ID == "kokafr"].replies.iloc[1].__getitem__(0).parent_id
d_comments[d_comments.PARENT_ID == "kokafr"].replies.iloc[1].__getitem__(0).is_root

False

# PIPELINE
## comments ingestion

In [7]:
def is_comment_forest_empty(comment_forest):
    try:
        comment_forest.__getitem__(0)
        return True
    except:
        return False

def get_replies_from_comment_forest(comment_forest):
    replies_list = {}
    for idx, _ in enumerate(comment_forest):
        replies_list.update({
            "id" : [comment_forest.__getitem__(idx).id], 
            "parent_id": [comment_forest.__getitem__(idx).parent_id],
            "created_utc" : [comment_forest.__getitem__(idx).created_utc],
            "body" : [comment_forest.__getitem__(idx).body],
            "score" : [comment_forest.__getitem__(idx).score],
            "permalink": [comment_forest.__getitem__(idx).permalink],                     # A permalink for the comment. Comment objects from the inbox have a context attribute instead.
            "replies" : [comment_forest.__getitem__(idx).replies],                        # Provides an instance of CommentForest.
            "submission" : [comment_forest.__getitem__(idx).submission],                  # Provides an instance of Submission. The submission that the comment belongs to.
            "submission_id": [comment_forest.__getitem__(idx).link_id],                   # The submission ID that the comment belongs to.
            "subreddit" : [comment_forest.__getitem__(idx).subreddit],                    # Provides an instance of Subreddit. The subreddit that the comment belongs to.
            "subreddit_id" : [comment_forest.__getitem__(idx).subreddit_id],              # The subreddit ID that the comment belongs to.
            "is_root" : [comment_forest.__getitem__(idx).is_root]

        })
    
    more_comments = pd.DataFrame.from_dict(replies_list)
    more_comments["timestamp"] = more_comments.created_utc.apply(get_datetime_from_timestamp)
    
    more_comments[["PARENT_TYPE", "PARENT_ID"]] = more_comments.apply(lambda x: x.parent_id.split("_"), axis=1, result_type='expand')
    more_comments.sort_values("timestamp", inplace=True)

    return more_comments

comments_tmp = comments_tmp.reset_index().set_index("id")
comments_tmp["IS_REPLIES_EMPTY"] = comments_tmp.replies.apply(is_comment_forest_empty)
comments_tmp["MORE_COMMENTS"] = comments_tmp[comments_tmp.IS_REPLIES_EMPTY].replies.swifter.apply(get_replies_from_comment_forest)
comments_tmp["MORE_COMMENTS_LEN"] = comments_tmp[comments_tmp.IS_REPLIES_EMPTY].MORE_COMMENTS.swifter.apply(lambda x: x.shape[0])

more_comments = pd.DataFrame()
for one_more in comments_tmp[comments_tmp.IS_REPLIES_EMPTY].MORE_COMMENTS.values:
    more_comments = pd.concat([
        more_comments, 
        one_more
    ])

col_to_drop = ["IS_REPLIES_EMPTY", "MORE_COMMENTS", "MORE_COMMENTS_LEN"]
comments = pd.concat([
    comments_tmp.reset_index().drop(columns=col_to_drop), more_comments
])
comments["_submission_"] = comments.submission.swifter.apply(lambda x: x.id)
comments.drop(columns=["replies", "submission", "subreddit"], inplace=True)

"""
    consistency checks:
        1) all submissions id (subsetting on submissions with comments) must be listed in the comments _submission_ field (parsed submission field)
        2) comments_tmp and more_comments must have same columns
"""
assert np.all(submissions[submissions.HAS_COMMENTS].id.sort_values().values.tolist() == comments._submission_.value_counts().index.sort_values().tolist()), "submissions id with commentes {} -> more_comments {}".format(submissions[submissions.HAS_COMMENTS].id.sort_values().values.tolist(), comments._submission_.value_counts().index.sort_values().tolist())

assert np.all(comments_tmp.reset_index().drop(columns=col_to_drop).columns == more_comments.columns), "comments_tmp {} -> more_comments {}".format(comments_tmp.reset_index().drop(columns=col_to_drop).columns.tolist(), more_comments.columns.tolist())

Pandas Apply: 100%|██████████| 79/79 [00:00<?, ?it/s]
C:\Users\fontanesio\Anaconda3\lib\site-packages\swifter\swifter.py:36: UserWarning: This pandas object has duplicate indices, and swifter may not be able to improve performance. Consider resetting the indices with `df.reset_index(drop=True)`.
  warnings.warn(
Pandas Apply: 100%|██████████| 307/307 [00:00<?, ?it/s]


# Analysis
## 

In [8]:
comments.head(2)

,id,parent_id,created_utc,body,score,permalink,submission_id,subreddit_id,is_root,timestamp,PARENT_TYPE,PARENT_ID,_submission_
0,ghrjlz2,t3_kok5cp,1.609539e+09,"Eat my dongus you fuckin nerd.\n\n*I am a bot,...",1,/r/wallstreetbets/comments/kok5cp/i_sexually_i...,t3_kok5cp,t5_2th52,True,2021-01-01 23:02:28,t3,kok5cp,kok5cp
1,ghrjnru,t3_kok5cp,1.609539e+09,"i renamed the post , it fits it better",-1,/r/wallstreetbets/comments/kok5cp/i_sexually_i...,t3_kok5cp,t5_2th52,True,2021-01-01 23:02:55,t3,kok5cp,kok5cp


In [9]:
for a, b, c in zip(more_comments.shape, comments_tmp.shape, comments.shape):
    print("more comm {}, comm {}, tot_comm {}, sum {} ".format(a, b, c, a+b))

more comm 79, comm 228, tot_comm 307, sum 307 
more comm 15, comm 17, tot_comm 13, sum 32 


In [10]:
comments_tmp[comments_tmp.submission == "kokdo8"].shape[0], more_comments[more_comments.submission == "kokdo8"].shape[0], comments_tmp[comments_tmp.submission == "kokdo8"].shape[0] + more_comments[more_comments.submission == "kokdo8"].shape[0]

(102, 35, 137)

In [11]:
submissions[submissions.id == "kokdo8"].COMMENTS_NUM.values[0], submissions[submissions.id == "kokdo8"].num_comments.values[0]

(102.0, 117)

# CHECKPOINT
    
## Export comments.parquet
## Export submissions.parquet 

In [12]:
comments.to_parquet("comments.parquet", engine="fastparquet") # .drop(columns=["replies", "submission", "subreddit"])

In [13]:
comments.head()

,id,parent_id,created_utc,body,score,permalink,submission_id,subreddit_id,is_root,timestamp,PARENT_TYPE,PARENT_ID,_submission_
0,ghrjlz2,t3_kok5cp,1.609539e+09,"Eat my dongus you fuckin nerd.\n\n*I am a bot,...",1,/r/wallstreetbets/comments/kok5cp/i_sexually_i...,t3_kok5cp,t5_2th52,True,2021-01-01 23:02:28,t3,kok5cp,kok5cp
1,ghrjnru,t3_kok5cp,1.609539e+09,"i renamed the post , it fits it better",-1,/r/wallstreetbets/comments/kok5cp/i_sexually_i...,t3_kok5cp,t5_2th52,True,2021-01-01 23:02:55,t3,kok5cp,kok5cp
2,ghrkap5,t3_kok7my,1.609539e+09,"Hi retards, it's me again with another chart. ...",37,/r/wallstreetbets/comments/kok7my/ryan_cohengm...,t3_kok7my,t5_2th52,True,2021-01-01 23:08:32,t3,kok7my,kok7my
3,ghrkgqf,t3_kokafr,1.609539e+09,"Eat my dongus you fuckin nerd.\n\n*I am a bot,...",8,/r/wallstreetbets/comments/kokafr/i_sexually_i...,t3_kokafr,t5_2th52,True,2021-01-01 23:09:58,t3,kokafr,kokafr
4,ghrkn71,t3_kokafr,1.609539e+09,"This is too gay, even for WSB. BAN",3,/r/wallstreetbets/comments/kokafr/i_sexually_i...,t3_kokafr,t5_2th52,True,2021-01-01 23:11:30,t3,kokafr,kokafr


In [22]:
comments = pd.read_parquet("comments.parquet", engine="fastparquet")
comments.set_index(["_submission_", "id"], inplace=True)
comments.head(2)

parent_id   created_utc  \
_submission_ id                                 
kok5cp       ghrjlz2  t3_kok5cp  1.609539e+09   
             ghrjnru  t3_kok5cp  1.609539e+09   

                                                                   body  \
_submission_ id                                                           
kok5cp       ghrjlz2  Eat my dongus you fuckin nerd.\n\n*I am a bot,...   
             ghrjnru             i renamed the post , it fits it better   

                      score  \
_submission_ id               
kok5cp       ghrjlz2      1   
             ghrjnru     -1   

                                                              permalink  \
_submission_ id                                                           
kok5cp       ghrjlz2  /r/wallstreetbets/comments/kok5cp/i_sexually_i...   
             ghrjnru  /r/wallstreetbets/comments/kok5cp/i_sexually_i...   

                     submission_id subreddit_id  is_root           timestamp  \
_submission_ id                                                                
kok5cp       ghrjlz2     t3_kok5cp     t5_2th52     True 2021-01-01 23:02:28   
             ghrjnru     t3_kok5cp     t5_2th52     True 2021-01-01 23:02:55   

                     PARENT_TYPE PARENT_ID  
_submission_ id                             
kok5cp       ghrjlz2          t3    kok5cp  
             ghrjnru          t3    kok5cp

In [16]:
submissions.to_parquet("submissions.parquet", engine="fastparquet")

In [21]:
submissions = pd.read_parquet("submissions.parquet", engine="fastparquet")
submissions.head(2)

,id,author,created_utc,datetime,domain,url,title,score,selftext,num_comments,num_crossposts,full_link,link_flair_text,COMMENTS_NUM,HAS_COMMENTS
index,,,,,,,,,,,,,,,
27,kok5cp,Lucky-Designer3469,1609538548,2021-01-01 23:02:28,self.wallstreetbets,https://www.reddit.com/r/wallstreetbets/commen...,I Sexually identify as a Gay Bear,1,i will admit it ima gay bear . Ever since I wa...,3,0,https://www.reddit.com/r/wallstreetbets/commen...,Meme,2.0,True
26,kok5r8,gotoptions_,1609538587,2021-01-01 23:03:07,i.redd.it,https://i.redd.it/i861jxb2ns861.jpg,A bet on you being even more retarded,1,,0,0,https://www.reddit.com/r/wallstreetbets/commen...,Meme,NaN,False


# Analysis
# Vader : Implementing sentiment indicators

In [46]:
submissions[submissions.COMMENTS_NUM > 0][["id", "COMMENTS_NUM"]].sort_values("COMMENTS_NUM")

,id,COMMENTS_NUM
index,,
15,kokijl,1.0
4,kokxko,1.0
8,kokryo,1.0
11,kokmnd,1.0
1,kol2v7,1.0
3,kol20h,2.0
5,kokxhq,2.0
27,kok5cp,2.0
0,kol442,2.0


In [92]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyser = SentimentIntensityAnalyzer()
# output : {'neg': 0.0, 'neu': 0.924, 'pos': 0.076, 'compound': 0.3182}
comments[["POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND"]] = comments.apply(lambda x: analyser.polarity_scores(x.body).values(), axis=1, result_type='expand')

In [49]:
comments[comments.index.get_level_values(0) == "kokdo8"].head()

parent_id   created_utc  \
_submission_ id                                  
kokdo8       ghrl8xk   t3_kokdo8  1.609539e+09   
             ghrlm3n   t3_kokdo8  1.609540e+09   
             ghrlpvo  t1_ghrlm3n  1.609540e+09   
             ghrlstu  t1_ghrlpvo  1.609540e+09   
             ghrlx1d   t3_kokdo8  1.609540e+09   

                                                                   body  \
_submission_ id                                                           
kokdo8       ghrl8xk  Finna suck my own cock in the back of this dum...   
             ghrlm3n                                          [deleted]   
             ghrlpvo                      I figured I’d get roasted lol   
             ghrlstu                                       I’m ready 😭🤧   
             ghrlx1d  University of North Wisconsinville majoring in...   

                      score  \
_submission_ id               
kokdo8       ghrl8xk     80   
             ghrlm3n     33   
             ghrlpvo      8   
             ghrlstu      4   
             ghrlx1d     67   

                                                              permalink  \
_submission_ id                                                           
kokdo8       ghrl8xk  /r/wallstreetbets/comments/kokdo8/dear_student...   
             ghrlm3n  /r/wallstreetbets/comments/kokdo8/dear_student...   
             ghrlpvo  /r/wallstreetbets/comments/kokdo8/dear_student...   
             ghrlstu  /r/wallstreetbets/comments/kokdo8/dear_student...   
             ghrlx1d  /r/wallstreetbets/comments/kokdo8/dear_student...   

                     submission_id subreddit_id  is_root           timestamp  \
_submission_ id                                                                
kokdo8       ghrl8xk     t3_kokdo8     t5_2th52     True 2021-01-01 23:16:50   
             ghrlm3n     t3_kokdo8     t5_2th52     True 2021-01-01 23:19:59   
             ghrlpvo     t3_kokdo8     t5_2th52    False 2021-01-01 23:20:54   
             ghrlstu     t3_kokdo8     t5_2th52    False 2021-01-01 23:21:37   
             ghrlx1d     t3_kokdo8     t5_2th52     True 2021-01-01 23:22:39   

                     PARENT_TYPE PARENT_ID  POLARITY_SCORE_NEG  \
_submission_ id                                                  
kokdo8       ghrl8xk          t3    kokdo8               0.325   
             ghrlm3n          t3    kokdo8               0.000   
             ghrlpvo          t1   ghrlm3n               0.000   
             ghrlstu          t1   ghrlpvo               0.292   
             ghrlx1d          t3    kokdo8               0.059   

                      POLARITY_SCORE_NEUT  POLARITY_SCORE_POS  \
_submission_ id                                                 
kokdo8       ghrl8xk                0.675               0.000   
             ghrlm3n                1.000               0.000   
             ghrlpvo                0.641               0.359   
             ghrlstu                0.472               0.236   
             ghrlx1d                0.941               0.000   

                      POLARITY_SCORE_COMPOUND  
_submission_ id                                
kokdo8       ghrl8xk                  -0.6428  
             ghrlm3n                   0.0000  
             ghrlpvo                   0.4215  
             ghrlstu                  -0.1531  
             ghrlx1d                  -0.2057

In [95]:
comments[["POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND"]].boxplot(figsize=(10, 10)) # , "score"

In [51]:
comments[["POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND"]].describe()

,POLARITY_SCORE_NEG,POLARITY_SCORE_NEUT,POLARITY_SCORE_POS,POLARITY_SCORE_COMPOUND
count,307.000000,307.000000,307.000000,307.000000
mean,0.084052,0.799906,0.116013,0.081977
std,0.150221,0.216775,0.171680,0.449794
min,0.000000,0.000000,0.000000,-0.929700
25%,0.000000,0.713000,0.000000,-0.115350
50%,0.000000,0.831000,0.048000,0.000000
75%,0.120000,1.000000,0.170000,0.440400
max,1.000000,1.000000,1.000000,0.973800


In [106]:
_GROUPBY_COL_ = "_submission_" # "_submission_" #"PARENT_ID"
_POLARITY_COL_ = "POLARITY_SCORE_COMPOUND"
_STATS_COL_  = "mean"


comments_stats = comments.reset_index()[[_GROUPBY_COL_, "POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND", "score"]].groupby(_GROUPBY_COL_).agg({min, max, sum, np.mean, np.std}).T
comments_stats.head()

_submission_             kok5cp    kok5vy    kok7my    kok8mp    kokafr  \
POLARITY_SCORE_NEG sum      0.0  0.339000  5.743000  1.701000  2.382000   
                   std      0.0  0.138769  0.111463  0.109182  0.453035   
                   mean     0.0  0.084750  0.075566  0.073957  0.297750   
                   max      0.0  0.290000  0.564000  0.288000  1.000000   
                   min      0.0  0.000000  0.000000  0.000000  0.000000   

_submission_                kokdo8    kokf03   kokhoh  kokijl    koklmq  \
POLARITY_SCORE_NEG sum   10.913000  0.240000  0.47400   0.049  2.133000   
                   std    0.136618  0.069282  0.19351     NaN  0.167738   
                   mean   0.079657  0.060000  0.07900   0.049  0.092739   
                   max    0.647000  0.120000  0.47400   0.049  0.643000   
                   min    0.000000  0.000000  0.00000   0.049  0.000000   

_submission_             kokmnd  kokryo    kokwhb    kokxhq  kokxko  kol20h  \
POLARITY_SCORE_NEG sum    0.524   0.041  0.877000  0.101000   0.041     0.0   
                   std      NaN     NaN  0.138722  0.058312     NaN     0.0   
                   mean   0.524   0.041  0.073083  0.033667   0.041     0.0   
                   max    0.524   0.041  0.429000  0.101000   0.041     0.0   
                   min    0.524   0.041  0.000000  0.000000   0.041     0.0   

_submission_             kol2v7    kol442  
POLARITY_SCORE_NEG sum    0.041  0.205000  
                   std      NaN  0.144957  
                   mean   0.041  0.102500  
                   max    0.041  0.205000  
                   min    0.041  0.000000

In [116]:
submissions.loc[:, f"{_POLARITY_COL_}_{ _STATS_COL_}"] = comments_stats.loc[comments_stats.index == (_POLARITY_COL_, _STATS_COL_)].T.sort_values((_POLARITY_COL_, _STATS_COL_), ascending=False)

In [117]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_GROUPBY_COL_ = "_submission_"
_POLARITY_COL_ = "POLARITY_SCORE_COMPOUND"
_STATS_COL_  = "mean"

analyser = SentimentIntensityAnalyzer()
"""
SentimentIntensityAnalyzer().polarity_scores()
    output : {'neg': 0.0, 'neu': 0.924, 'pos': 0.076, 'compound': 0.3182}
"""
comments[["POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND"]] = comments.apply(lambda x: analyser.polarity_scores(x.body).values(), axis=1, result_type='expand')

comments_stats = comments.reset_index()[[_GROUPBY_COL_, "POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND", "score"]].groupby(_GROUPBY_COL_).agg({min, max, sum, np.mean, np.std}).T

submissions.loc[:, f"{_POLARITY_COL_}_{ _STATS_COL_}"] = comments_stats.loc[comments_stats.index == (_POLARITY_COL_, _STATS_COL_)].T.sort_values((_POLARITY_COL_, _STATS_COL_), ascending=False)

In [97]:
submissions = comments_stats.loc[comments_stats.index.get_level_values(1) == _STATS_COL_].sort_values(_STATS_COL_, ascending=False).merge(
    submissions, left_index=True, right_index=True
)

submissions = submissions[submissions.COMMENTS_NUM > 2]

KeyError: 'mean'

In [115]:
tt = comments_stats.T.loc[comments_stats.columns.get_level_values(1) == _STATS_COL_].T[_POLARITY_COL_].sort_values(_STATS_COL_, ascending=False).merge(
    submissions[["id", "COMMENTS_NUM"]], left_index=True, right_on="id"
).set_index("id")

tt = tt[tt.COMMENTS_NUM > 2]

In [123]:
tt

,mean,COMMENTS_NUM
id,,
kok5vy,0.371075,4.0
kokhoh,0.147133,5.0
kokwhb,0.122100,8.0
kok7my,0.115938,54.0
kok8mp,0.066191,17.0
kokdo8,0.047312,102.0
koklmq,0.026639,16.0
kokf03,0.006400,3.0
kokafr,-0.128287,6.0


In [124]:
np.mean(comments[comments.index.get_level_values(0) == "kokhoh"].POLARITY_SCORE_COMPOUND)

0.14713333333333334

In [125]:
_TOP_ = 3
_LAST_ = -3

In [126]:
submissions.set_index("id").loc[tt.index[:_TOP_]].title

id
kok5vy                     FDX 8Jan weeklies for discussion
kokhoh    I need Help I Sexually identify as a WSB Bear ...
kokwhb    BABA bout to explode (full disclosure i own BABA)
Name: title, dtype: object

In [134]:
comments.loc[comments.index.get_level_values(0).isin(tt.index[:_TOP_])][["body", _POLARITY_COL_, "score"]]

body  \
_submission_ id                                                           
kokhoh       ghrlqpp  Eat my dongus you fuckin nerd.\n\n*I am a bot,...   
             ghrlqqy  Nobody tell him\n\n*I am a bot, and this actio...   
             ghrmedi                                                Gay   
             ghrmgat                        i need help , convert me ..   
             ghrn143                             Ban, you larp too much   
kokwhb       ghrofko           Damn, use a ruler or something next time   
             ghroxdl  Mind to leverage why it will explode in near f...   
             ghrp3k2  Basically the Amazon of China, and china’s eco...   
             ghrpzfa  it’s got a PE of about 30 vs Amazon’s of over ...   
             ghrqf1g  I can see long term. I am talking about short ...   
             ghrqfmv                                   Strike and date?   
             ghrqtx0  i’m thinking it will be up 5-10% within a week...   
             ghrr4k2  I bought 2 at 224. Sold it Thur night. That's ...   
kok5vy       ghrrqeq  Yeah it’s dumb. Weeklies are almost always dum...   
             ghrsyv3  When you’re buying something just think hmmm I...   
             ghrvw6f  Wouldn’t do weeklies, give it a few months. Sa...   
             gi3pjby  Thank you all for that advice.  Glad i listene...   
kokhoh       ghrmgat                        i need help , convert me ..   
kokwhb       ghrpzfa  it’s got a PE of about 30 vs Amazon’s of over ...   
             ghrqf1g  I can see long term. I am talking about short ...   
             ghrqtx0  i’m thinking it will be up 5-10% within a week...   
             ghrr4k2  I bought 2 at 224. Sold it Thur night. That's ...   

                      POLARITY_SCORE_COMPOUND  score  
_submission_ id                                       
kokhoh       ghrlqpp                   0.3182      4  
             ghrlqqy                   0.3182      1  
             ghrmedi                   0.0000      2  
             ghrmgat                   0.4019      0  
             ghrn143                  -0.5574      1  
kokwhb       ghrofko                  -0.4019      3  
             ghroxdl                   0.0000      1  
             ghrp3k2                   0.4939      1  
             ghrpzfa                   0.3818      1  
             ghrqf1g                  -0.3400      1  
             ghrqfmv                  -0.1280      1  
             ghrqtx0                   0.0000      1  
             ghrr4k2                   0.7088      1  
kok5vy       ghrrqeq                  -0.2075      2  
             ghrsyv3                   0.0000      1  
             ghrvw6f                   0.8742      5  
             gi3pjby                   0.8176      1  
kokhoh       ghrmgat                   0.4019      0  
kokwhb       ghrpzfa                   0.3818      1  
             ghrqf1g                  -0.3400      1  
             ghrqtx0                   0.0000      1  
             ghrr4k2                   0.7088      1

In [135]:
submissions.set_index("id").loc[tt.index[_LAST_:]][["title", "score"]]

,title,score
id,,
koklmq,GT on track to rocket for tendies. To the Moon...,1
kokf03,TSLA Future Model Lineup You S3XY B1TCH(ES)🚀 🚀 🚀,1
kokafr,I Sexually identify as a WSB Bear,1


In [136]:
comments.loc[comments.index.get_level_values(0).isin(tt.index[:_LAST_])][["body", _POLARITY_COL_,  "score"]]

body  \
_submission_ id                                                           
kok7my       ghrkap5  Hi retards, it's me again with another chart. ...   
kok8mp       ghrkp18  You’re missing the extra comma that you would’...   
kok7my       ghrkqdi  Looking forward to seeing shorts get lit up 🩳 🔥 🚀   
             ghrkuwr                                     good dd thanks   
             ghrl10r                                   Doing God's work   
...                                                                 ...   
kokdo8       ghssl1d                                          Lol dirty   
             ghtu5kp                                     Meth is a drug   
kok8mp       ghujf1x  Maybe. But he's made a killing on GME I'm pret...   
             ghukm5r  Did he really make any gains on gamestop? I he...   
             ghul8ky  Very possible. I don't follow all his moves, p...   

                      POLARITY_SCORE_COMPOUND  score  
_submission_ id                                       
kok7my       ghrkap5                   0.0000     37  
kok8mp       ghrkp18                  -0.7906     45  
kok7my       ghrkqdi                  -0.3400     13  
             ghrkuwr                   0.7003      5  
             ghrl10r                   0.0000      8  
...                                       ...    ...  
kokdo8       ghssl1d                  -0.0258      2  
             ghtu5kp                   0.0000      1  
kok8mp       ghujf1x                   0.5927      1  
             ghukm5r                   0.6282      1  
             ghul8ky                  -0.1779      1  

[258 rows x 3 columns]

# PIPELINE
## COMMMENTS TAGGING AND EXTRACT SENTIMENT

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from preproc import *
import pandas as pd
import numpy as np
from psaw import PushshiftAPI

import spacy
import plotly.express as px
from collections import Counter

nlp = spacy.load("en_core_web_lg")

In [4]:
def add_sentiment_polarity():
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    """
    SentimentIntensityAnalyzer().polarity_scores()
        output : {'neg': 0.0, 'neu': 0.924, 'pos': 0.076, 'compound': 0.3182}
    """

    _GROUPBY_COL_ = "_submission_"
    _POLARITY_COL_ = "POLARITY_SCORE_COMPOUND"
    _STATS_COL_  = "mean"
    
    _TOKEN_TYPES_ = ["PROPN", "VERB"]

    comments = pd.read_parquet("comments.parquet", engine="fastparquet")
    comments.set_index(["_submission_", "id"], inplace=True)

    submissions = pd.read_parquet("submissions.parquet", engine="fastparquet")
    submissions.set_index("id", inplace=True)

    print(submissions.shape, comments.shape)

    analyser = SentimentIntensityAnalyzer()
    comments[["POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND"]] = comments.apply(lambda x: analyser.polarity_scores(x.body).values(), axis=1, result_type='expand')

    comments_stats = comments.reset_index()[[_GROUPBY_COL_, "POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND", "score"]].groupby(_GROUPBY_COL_).agg({min, max, sum, np.mean, np.std}).T

    comments_stats_tmp = comments_stats.loc[comments_stats.index == (_POLARITY_COL_, _STATS_COL_)].T
    comments_stats_tmp.columns = comments_stats_tmp.columns.droplevel()
    submissions.loc[comments_stats_tmp.index, _STATS_COL_] = comments_stats_tmp
    submissions.rename(columns={_STATS_COL_ : f"{_POLARITY_COL_}_{_STATS_COL_}"}, inplace=True)


    submissions["DOC"] = submissions.title.apply(nlp)
    submissions["DOC_TOKEN"] = submissions.DOC.apply(lambda x: [{"token_text" : token.text, "token_pos_" : token.pos_, "token_dep_" : token.dep_, "token_lemma_" : token.lemma_} for token in x])
    submissions[["DOC_TOKEN"]] = submissions[["DOC_TOKEN"]].apply(lambda x: [token.update({"id" : x.name}) or token for token in x[0]], axis=1)

    submissions["DOC_TOKEN_TMP"] = submissions["DOC_TOKEN"].apply(lambda x: pd.DataFrame(x))
    tokens_submissions = pd.DataFrame()
    for token in submissions["DOC_TOKEN_TMP"].values:
        tokens_submissions = pd.concat([tokens_submissions, token])
    submissions.drop(columns=["DOC_TOKEN_TMP"], inplace=True)


    comments["DOC"] = comments.body.apply(nlp)
    comments["DOC_TOKEN"] = comments.DOC.apply(lambda x: [{"token_text" : token.text, "token_pos_" : token.pos_, "token_dep_" : token.dep_, "token_lemma_" : token.lemma_} for token in x])
    comments[["DOC_TOKEN"]] = comments[["DOC_TOKEN"]].apply(lambda x: [token.update({"_submission_" : x.name[0], "comment_id" : x.name[1]}) or token for token in x[0]], axis=1)

    comments["DOC_TOKEN_TMP"] = comments["DOC_TOKEN"].apply(lambda x: pd.DataFrame(x))
    tokens_comments = pd.DataFrame()
    for token in comments["DOC_TOKEN_TMP"].values:
        tokens_comments = pd.concat([tokens_comments, token])
    comments.drop(columns=["DOC_TOKEN_TMP"], inplace=True)


    for token_type in _TOKEN_TYPES_:
        submissions.loc[:, f"token_{token_type}"] = tokens_comments[tokens_comments.token_pos_==token_type].groupby("_submission_").token_lemma_.apply(lambda x: Counter(x).most_common())
    
    return submissions, comments, tokens_submissions, tokens_comments

In [8]:
submissions, comments, tokens_submissions, tokens_comments = add_sentiment_polarity()
print(submissions.shape, comments.shape, tokens_submissions.shape, tokens_comments.shape)

(28, 14) (307, 11)
(28, 19) (307, 17) (268, 5) (8911, 6)


# ANALYSIS
## NLP

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from preproc import *
import pandas as pd
import numpy as np
from psaw import PushshiftAPI

import spacy
import plotly.express as px
from collections import Counter

nlp = spacy.load("en_core_web_lg")

In [2]:
comments = pd.read_parquet("comments.parquet", engine="fastparquet")
comments.set_index(["_submission_", "id"], inplace=True)
comments.head(1)

,,parent_id,created_utc,body,score,permalink,submission_id,subreddit_id,is_root,timestamp,PARENT_TYPE,PARENT_ID
_submission_,id,,,,,,,,,,,
kok5cp,ghrjlz2,t3_kok5cp,1.609539e+09,"Eat my dongus you fuckin nerd.\n\n*I am a bot,...",1,/r/wallstreetbets/comments/kok5cp/i_sexually_i...,t3_kok5cp,t5_2th52,True,2021-01-01 23:02:28,t3,kok5cp


In [3]:
submissions = pd.read_parquet("submissions.parquet", engine="fastparquet")
submissions.set_index("id", inplace=True)
submissions.head(1)

,author,created_utc,datetime,domain,url,title,score,selftext,num_comments,num_crossposts,full_link,link_flair_text,COMMENTS_NUM,HAS_COMMENTS
id,,,,,,,,,,,,,,
kok5cp,Lucky-Designer3469,1609538548,2021-01-01 23:02:28,self.wallstreetbets,https://www.reddit.com/r/wallstreetbets/commen...,I Sexually identify as a Gay Bear,1,i will admit it ima gay bear . Ever since I wa...,3,0,https://www.reddit.com/r/wallstreetbets/commen...,Meme,2.0,True


In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_GROUPBY_COL_ = "_submission_"
_POLARITY_COL_ = "POLARITY_SCORE_COMPOUND"
_STATS_COL_  = "mean"

analyser = SentimentIntensityAnalyzer()
"""
SentimentIntensityAnalyzer().polarity_scores()
    output : {'neg': 0.0, 'neu': 0.924, 'pos': 0.076, 'compound': 0.3182}
"""
comments[["POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND"]] = comments.apply(lambda x: analyser.polarity_scores(x.body).values(), axis=1, result_type='expand')

comments_stats = comments.reset_index()[[_GROUPBY_COL_, "POLARITY_SCORE_NEG", "POLARITY_SCORE_NEUT", "POLARITY_SCORE_POS", "POLARITY_SCORE_COMPOUND", "score"]].groupby(_GROUPBY_COL_).agg({min, max, sum, np.mean, np.std}).T

"""
submissions.loc[:, f"{_POLARITY_COL_}_{ _STATS_COL_}"] = comments_stats.loc[comments_stats.index == (_POLARITY_COL_, _STATS_COL_)].T.sort_values((_POLARITY_COL_, _STATS_COL_), ascending=False)
"""

comments_stats_tmp = comments_stats.loc[comments_stats.index == (_POLARITY_COL_, _STATS_COL_)].T
comments_stats_tmp.columns = comments_stats_tmp.columns.droplevel()
submissions.loc[comments_stats_tmp.index, _STATS_COL_] = comments_stats_tmp
submissions.rename(columns={_STATS_COL_ : f"{_POLARITY_COL_}_{_STATS_COL_}"}, inplace=True)

In [5]:
submissions["DOC"] = submissions.title.apply(nlp)
submissions["DOC_TOKEN"] = submissions.DOC.apply(lambda x: [{"token_text" : token.text, "token_pos_" : token.pos_, "token_dep_" : token.dep_, "token_lemma_" : token.lemma_} for token in x])
submissions[["DOC_TOKEN"]] = submissions[["DOC_TOKEN"]].apply(lambda x: [token.update({"id" : x.name}) or token for token in x[0]], axis=1)

submissions["DOC_TOKEN_TMP"] = submissions["DOC_TOKEN"].apply(lambda x: pd.DataFrame(x))
tokens_submissions = pd.DataFrame()
for token in submissions["DOC_TOKEN_TMP"].values:
    tokens_submissions = pd.concat([tokens_submissions, token])
submissions.drop(columns=["DOC_TOKEN_TMP"], inplace=True)
tokens_submissions.head()

,token_text,token_pos_,token_dep_,token_lemma_,id
0,I,PRON,nsubj,I,kok5cp
1,Sexually,ADV,advmod,sexually,kok5cp
2,identify,VERB,ROOT,identify,kok5cp
3,as,ADP,prep,as,kok5cp
4,a,DET,det,a,kok5cp


In [6]:
comments["DOC"] = comments.body.apply(nlp)
comments["DOC_TOKEN"] = comments.DOC.apply(lambda x: [{"token_text" : token.text, "token_pos_" : token.pos_, "token_dep_" : token.dep_, "token_lemma_" : token.lemma_} for token in x])
comments[["DOC_TOKEN"]] = comments[["DOC_TOKEN"]].apply(lambda x: [token.update({"_submission_" : x.name[0], "comment_id" : x.name[1]}) or token for token in x[0]], axis=1)

comments["DOC_TOKEN_TMP"] = comments["DOC_TOKEN"].apply(lambda x: pd.DataFrame(x))
tokens_comments = pd.DataFrame()
for token in comments["DOC_TOKEN_TMP"].values:
    tokens_comments = pd.concat([tokens_comments, token])
comments.drop(columns=["DOC_TOKEN_TMP"], inplace=True)
tokens_comments.head()

,token_text,token_pos_,token_dep_,token_lemma_,_submission_,comment_id
0,Eat,VERB,ROOT,eat,kok5cp,ghrjlz2
1,my,PRON,poss,my,kok5cp,ghrjlz2
2,dongus,NOUN,dobj,dongus,kok5cp,ghrjlz2
3,you,PRON,nsubj,you,kok5cp,ghrjlz2
4,fuckin,ADJ,nsubj,fuckin,kok5cp,ghrjlz2


In [7]:
token_types = ["PROPN", "VERB"]
for token_type in token_types:
    submissions.loc[:, f"token_{token_type}"] = tokens_comments[tokens_comments.token_pos_==token_type].groupby("_submission_").token_lemma_.apply(lambda x: Counter(x).most_common())

In [8]:
tokens_submissions.shape, tokens_comments.shape

((268, 5), (8911, 6))

In [19]:
submissions[
    (submissions[f"{_POLARITY_COL_}_{_STATS_COL_}"].notnull())
    &
    (submissions.COMMENTS_NUM > 2.0)
    ].sort_values(f"{_POLARITY_COL_}_{_STATS_COL_}", ascending=False)[["POLARITY_SCORE_COMPOUND_mean", "COMMENTS_NUM", "token_PROPN", "token_VERB"]]

,POLARITY_SCORE_COMPOUND_mean,COMMENTS_NUM,token_PROPN,token_VERB
id,,,,
kok5vy,0.371075,4.0,"[(Jan, 2), (FDX, 1), (Walmart, 1), (Black, 1),...","[(do, 3), (be, 3), (give, 2), (’, 1), (buy, 1)..."
kokhoh,0.147133,5.0,NaN,"[(perform, 2), (contact, 2), (have, 2), (need,..."
kokwhb,0.122100,8.0,"[(Amazon, 3), (USs, 2), (Jan, 2), (Thur, 2), (...","[(explode, 3), (outpace, 3), (see, 3), (go, 3)..."
kok7my,0.115938,54.0,"[(RC, 17), (GME, 14), (Sherman, 11), (Cohen, 1...","[(be, 43), (buy, 24), (have, 21), (get, 20), (..."
kok8mp,0.066191,17.0,"[(November, 4), (🌈, 3), (Tesla, 3), (🏳, 2), (️...","[(sell, 9), (buy, 8), (make, 5), (think, 5), (..."
kokdo8,0.047312,102.0,"[(University, 8), (😎, 7), (PLTR, 6), (CS, 5), ...","[(get, 21), (be, 21), (work, 19), (do, 18), (m..."
koklmq,0.026639,16.0,"[(EPS, 4), (GT, 2), (TESLA, 2), (Tesla, 2), (🚀...","[(drive, 4), (do, 3), (’m, 3), (fly, 2), (ban,..."
kokf03,0.006400,3.0,"[(Sir, 1), (Tesla\n\nthis, 1)]","[(want, 3), (need, 3), (buy, 2), (provide, 2),..."
kokafr,-0.128287,6.0,"[(Karen, 2), (WSB, 1)]","[(adopt, 2), (eat, 1), (perform, 1), (contact,..."


In [54]:
Counter(tokens_comments[tokens_comments.token_pos_=="PROPN"].token_lemma_.values).most_common(20)

[('GME', 21),
 ('RC', 17),
 ('Sherman', 11),
 ('Cohen', 10),
 ('University', 8),
 ('PLTR', 8),
 ('Jan', 8),
 ('🚀', 7),
 ('😎', 7),
 ('YOLO', 6),
 ('🌈', 5),
 ('CS', 5),
 ('Tesla', 5),
 ('CNBC', 5),
 ('Beloit', 5),
 ('November', 5),
 ('🏳', 4),
 ('️\u200d', 4),
 ('Flair', 4),
 ('Guide](https://www.reddit.com', 4)]

In [60]:
tokens_comments[tokens_comments.token_pos_=="PROPN"].groupby("_submission_").token_lemma_.apply(lambda x: Counter(x).most_common())

_submission_
kok5vy    [(Jan, 2), (FDX, 1), (Walmart, 1), (Black, 1),...
kok7my    [(RC, 17), (GME, 14), (Sherman, 11), (Cohen, 1...
kok8mp    [(November, 4), (🌈, 3), (Tesla, 3), (🏳, 2), (️...
kokafr                               [(Karen, 2), (WSB, 1)]
kokdo8    [(University, 8), (😎, 7), (PLTR, 6), (CS, 5), ...
kokf03                       [(Sir, 1), (Tesla\n\nthis, 1)]
kokijl    [(resubmitting.\n\nPlease, 1), (banned.\n\n[Su...
koklmq    [(EPS, 4), (GT, 2), (TESLA, 2), (Tesla, 2), (🚀...
kokryo    [(YOLO, 2), (flair%3AYOLO&restrict_sr, 1), (on...
kokwhb    [(Amazon, 3), (USs, 2), (Jan, 2), (Thur, 2), (...
kokxhq                                  [(GME, 2), (RH, 1)]
kokxko    [(YOLO, 2), (flair%3AYOLO&restrict_sr, 1), (on...
kol20h                        [(🚀, 3), (GME, 1), (PLTR, 1)]
kol2v7    [(YOLO, 2), (flair%3AYOLO&restrict_sr, 1), (on...
kol442                                           [(Ban, 1)]
Name: token_lemma_, dtype: object

In [84]:
token_types = ["PROPN", "VERB"]
for token_type in token_types:
    submissions.loc[:, f"token_{token_type}"] = tokens_comments[tokens_comments.token_pos_==token_type].groupby("_submission_").token_lemma_.apply(lambda x: Counter(x).most_common())

In [88]:
submissions.head(1)

,author,created_utc,datetime,domain,url,title,score,selftext,num_comments,num_crossposts,full_link,link_flair_text,COMMENTS_NUM,HAS_COMMENTS,DOC,DOC_TOKEN,token_lemma_,token_PROPN,token_VERB
kok5vy,SirLouisI,1609538598,2021-01-01 23:03:18,self.wallstreetbets,https://www.reddit.com/r/wallstreetbets/commen...,FDX 8Jan weeklies for discussion,1,What do you degens think of FDX weeklies for n...,5,0,https://www.reddit.com/r/wallstreetbets/commen...,Discussion,4.0,True,"(FDX, 8Jan, weeklies, for, discussion)","[{'token_text': 'FDX', 'token_pos_': 'PROPN', ...","[(Jan, 2), (FDX, 1), (Walmart, 1), (Black, 1),...","[(Jan, 2), (FDX, 1), (Walmart, 1), (Black, 1),...","[(do, 3), (be, 3), (give, 2), (’, 1), (buy, 1)..."


In [91]:
Counter(tokens_comments[(tokens_comments._submission_ == submissions.index[1]) & (tokens_comments.token_pos_=="VERB")].token_lemma_.values).most_common()

[('be', 43),
 ('buy', 24),
 ('have', 21),
 ('get', 20),
 ('know', 20),
 ('say', 14),
 ('go', 13),
 ('make', 12),
 ('want', 12),
 ('see', 11),
 ('’', 10),
 ('sell', 10),
 ('do', 9),
 ('use', 9),
 ('look', 7),
 ('think', 7),
 ('come', 7),
 ('read', 7),
 ('try', 5),
 ('save', 5),
 ('notice', 4),
 ('change', 4),
 ('jump', 4),
 ('ask', 4),
 ('pull', 4),
 ('dip', 4),
 ('dig', 4),
 ('increase', 4),
 ('listen', 4),
 ('manage', 4),
 ('answer', 4),
 ('react', 3),
 ('suspect', 3),
 ('drop', 3),
 ('put', 3),
 ('seem', 3),
 ('beware', 3),
 ('hope', 3),
 ('happen', 3),
 ('quote', 3),
 ('surround', 3),
 ('step', 3),
 ('consider', 3),
 ('lean', 3),
 ('let', 2),
 ('settle', 2),
 ('hold', 2),
 ('suggest', 2),
 ('wait', 2),
 ('up', 2),
 ('handle', 2),
 ('start', 2),
 ('release', 2),
 ('end', 2),
 ('rocket', 2),
 ('cause', 2),
 ('stop', 2),
 ('purchase', 2),
 ('believe', 2),
 ('soar', 2),
 ('stick', 2),
 ('last', 2),
 ('need', 2),
 ('hand', 2),
 ('refer', 2),
 ('reduce', 2),
 ('take', 2),
 ('assign', 2),


In [69]:
comments[comments.index.get_level_values(0) == submissions.index[1]].body

_submission_  id     
kok7my        ghrkap5    Hi retards, it's me again with another chart. ...
              ghrkqdi    Looking forward to seeing shorts get lit up 🩳 🔥 🚀
              ghrkuwr                                       good dd thanks
              ghrl10r                                     Doing God's work
              ghrlhzv    That whole option settlement didn’t really cha...
                                               ...                        
              ghrvl34    Sry phone glitched out and posted multiple com...
              ghs0cbl    Paging u/Uberkikz11 for his views on what the ...
              ghs1cwr    I was at GME for close to an hour doing some D...
              ghsdje9    Same here, that “scared” emoji brings to mind ...
              ghsds0e    Lord Cohen knows that he can buy from the  🧻 🙌...
Name: body, Length: 76, dtype: object

In [48]:
tokens[~(tokens.token_pos_=="PROPN")].merge(submissions[["title", "COMMENTS_NUM"]], left_index=True, right_index=True)

,token_text,token_pos_,token_dep_,id,title,COMMENTS_NUM
index,,,,,,
27,I,PRON,nsubj,kok5cp,I Sexually identify as a Gay Bear,2.0
26,A,DET,det,kok5r8,A bet on you being even more retarded,NaN
23,$,SYM,nummod,kok8mp,$TSLA puts gain in 2020,17.0
22,I,PRON,nsubj,kokafr,I Sexually identify as a WSB Bear,6.0
21,Dear,ADJ,amod,kokdo8,Dear student retards...,102.0
20,How,ADV,advmod,kokdux,How an wsb autistic is born....,NaN
16,I,PRON,nsubj,kokhoh,I need Help I Sexually identify as a WSB Bear ...,5.0
15,Following,VERB,ROOT,kokijl,Following up on the Amzn 🎄 🚀 post. Look at Jan...,1.0
14,Rise,VERB,ROOT,kokikg,Rise up! Take ur stim to WAR!,NaN


In [9]:
import spacy
from nltk import Tree

nlp = spacy.load("en_core_web_lg")

def to_nltk_tree(node):
    if node.n_lefts + node.n_rights > 0:
        return Tree(node.orth_, [to_nltk_tree(child) for child in node.children])
    else:
        return node.orth_

In [10]:
tree = {}
for sub_id, title in zip(submissions.index.to_list(), submissions.title.values):
    doc = nlp(title)
    try:
        print(sub_id, title)
        tree.update({sub_id : [to_nltk_tree(sent.root) for sent in doc.sents]})
        [to_nltk_tree(sent.root).pretty_print() for sent in doc.sents]
    except Exception as e:
        print(e)
        print(sub_id, title)

kok5cp I Sexually identify as a Gay Bear
    identify             
  _____|__________        
 |     |          as     
 |     |          |       
 |     |         Bear    
 |     |       ___|____   
 I  Sexually  a       Gay

kok5r8 A bet on you being even more retarded
      bet                  
  _____|___________         
 |     |           on      
 |     |           |        
 |     |         being     
 |     |       ____|____    
 |     |      |        more
 |     |      |         |   
 A  retarded you       even

kok5vy FDX 8Jan weeklies for discussion
    weeklies           
  _____|_________       
 |     |        for    
 |     |         |      
FDX   8Jan   discussion

kok7my Ryan Cohen-GME confirmation bias
                  bias          
      _____________|_____        
     |                  GME     
     |         __________|____   
confirmation Ryan      Cohen  - 

kok8mp $TSLA puts gain in 2020
     puts     
  ____|____    
 |   TSLA  in 
 |    |    |   
gain  $

In [11]:
from spacy import displacy


In [15]:
displacy.render(submissions.loc["kok5vy"].DOC, style="dep")

In [12]:
submissions.loc["kok5vy"].DOC

FDX 8Jan weeklies for discussion

In [22]:
comments[comments.index.get_level_values(0) == "kok5vy"].DOC.apply(lambda x: (displacy.render(x, style="ent"), displacy.render(x, style="dep")))

C:\Users\fontanesio\Anaconda3\lib\site-packages\spacy\displacy\__init__.py:189: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


_submission_  id     
kok5vy        ghrrqeq    (None, None)
              ghrsyv3    (None, None)
              ghrvw6f    (None, None)
              gi3pjby    (None, None)
Name: DOC, dtype: object

In [18]:
comments.head()

parent_id   created_utc  \
_submission_ id                                 
kok5cp       ghrjlz2  t3_kok5cp  1.609539e+09   
             ghrjnru  t3_kok5cp  1.609539e+09   
kok7my       ghrkap5  t3_kok7my  1.609539e+09   
kokafr       ghrkgqf  t3_kokafr  1.609539e+09   
             ghrkn71  t3_kokafr  1.609539e+09   

                                                                   body  \
_submission_ id                                                           
kok5cp       ghrjlz2  Eat my dongus you fuckin nerd.\n\n*I am a bot,...   
             ghrjnru             i renamed the post , it fits it better   
kok7my       ghrkap5  Hi retards, it's me again with another chart. ...   
kokafr       ghrkgqf  Eat my dongus you fuckin nerd.\n\n*I am a bot,...   
             ghrkn71                 This is too gay, even for WSB. BAN   

                      score  \
_submission_ id               
kok5cp       ghrjlz2      1   
             ghrjnru     -1   
kok7my       ghrkap5     37   
kokafr       ghrkgqf      8   
             ghrkn71      3   

                                                              permalink  \
_submission_ id                                                           
kok5cp       ghrjlz2  /r/wallstreetbets/comments/kok5cp/i_sexually_i...   
             ghrjnru  /r/wallstreetbets/comments/kok5cp/i_sexually_i...   
kok7my       ghrkap5  /r/wallstreetbets/comments/kok7my/ryan_cohengm...   
kokafr       ghrkgqf  /r/wallstreetbets/comments/kokafr/i_sexually_i...   
             ghrkn71  /r/wallstreetbets/comments/kokafr/i_sexually_i...   

                     submission_id subreddit_id  is_root           timestamp  \
_submission_ id                                                                
kok5cp       ghrjlz2     t3_kok5cp     t5_2th52     True 2021-01-01 23:02:28   
             ghrjnru     t3_kok5cp     t5_2th52     True 2021-01-01 23:02:55   
kok7my       ghrkap5     t3_kok7my     t5_2th52     True 2021-01-01 23:08:32   
kokafr       ghrkgqf     t3_kokafr     t5_2th52     True 2021-01-01 23:09:58   
             ghrkn71     t3_kokafr     t5_2th52     True 2021-01-01 23:11:30   

                     PARENT_TYPE PARENT_ID  POLARITY_SCORE_NEG  \
_submission_ id                                                  
kok5cp       ghrjlz2          t3    kok5cp               0.000   
             ghrjnru          t3    kok5cp               0.000   
kok7my       ghrkap5          t3    kok7my               0.000   
kokafr       ghrkgqf          t3    kokafr               0.000   
             ghrkn71          t3    kokafr               0.382   

                      POLARITY_SCORE_NEUT  POLARITY_SCORE_POS  \
_submission_ id                                                 
kok5cp       ghrjlz2                0.924               0.076   
             ghrjnru                0.734               0.266   
kok7my       ghrkap5                1.000               0.000   
kokafr       ghrkgqf                0.924               0.076   
             ghrkn71                0.618               0.000   

                      POLARITY_SCORE_COMPOUND  \
_submission_ id                                 
kok5cp       ghrjlz2                   0.3182   
             ghrjnru                   0.4404   
kok7my       ghrkap5                   0.0000   
kokafr       ghrkgqf                   0.3182   
             ghrkn71                  -0.6523   

                                                                    DOC  \
_submission_ id                                                           
kok5cp       ghrjlz2  (Eat, my, dongus, you, fuckin, nerd.\n\n*I, am...   
             ghrjnru   (i, renamed, the, post, ,, it, fits, it, better)   
kok7my       ghrkap5  (Hi, retards, ,, it, 's, me, again, with, anot...   
kokafr       ghrkgqf  (Eat, my, dongus, you, fuckin, nerd.\n\n*I, am...   
             ghrkn71    (This, is, too, gay, ,, even, for, WSB, ., BAN)   

                                                          

In [81]:
BLACKLIST = ['ev', 'covid', 'etf', 'nyse', 'sec', 'spac', 'fda',
             'fed', 'treasury', 'eu', 'cnbc', 'faq', 'company',
             "🩳", "🚀", "wsb", "university of north wisconsinville",
             "university of michigan ross"]

def get_orgs(text):
    # process the text with our SpaCy model to get named entities
    doc = nlp(text)
    # initialize list to store identified organizations
    org_list = []
    for entity in doc.ents:
        # here we modify the original code to check that entity text is not equal to one of our 'blacklisted' organizations
        # (we also add .lower() to lowercase the text, this allows us to match both 'nyse' and 'NYSE' with just 'nyse')
        if entity.label_ == 'ORG' and entity.text.lower() not in BLACKLIST:
            org_list.append(entity.text)
    # if organization is identified more than once it will appear multiple times in list
    # we use set() to remove duplicates then convert back to list
    org_list = list(set(org_list))
    return org_list

In [82]:
comments['organizations'] = comments['body'].apply(get_orgs)

comments["HAS_ORG"] = comments.organizations.apply(lambda x: True if len(x) > 0 else False)
comments[comments["HAS_ORG"]].organizations

_submission_  id     
kok7my        ghrkap5                  [NYE]
              ghrlpdm                  [GME]
kokdo8        ghrm8i4             [CS, SFSU]
kok7my        ghrmf8p              [RC, ITM]
koklmq        ghrncbz                  [AMD]
                                ...         
kokdo8        ghrxelo                  [CJS]
              ghryyku                  [MKE]
kok7my        ghs1cwr          [GME, Switch]
kokdo8        ght2588    [Molecular Biology]
kok8mp        ghujf1x           [GME, Tesla]
Name: organizations, Length: 72, dtype: object

In [83]:
orgs = comments['organizations'].to_list()
orgs = [org for sublist in orgs for org in sublist]

org_freq = Counter(orgs)
org_freq.most_common(10)

[('GME', 17),
 ('RC', 9),
 ('PLTR', 6),
 ('Tesla', 5),
 ('CS', 4),
 ('ITM', 3),
 ('Amazon', 2),
 ('🏳️\u200d', 2),
 ('EPS', 2),
 ('EBIT', 2)]

In [85]:
txt = comments[comments.index.get_level_values(1) == "ghrncbz"].body.iloc[0]
txt

'Calls on a fucking tire company? LMAO. I guess I have done dumber shit than this (shorting $AMD before it jumped to $80)'

In [89]:
comments[comments.index.get_level_values(1) == "ghrncbz"].POLARITY_SCORE_COMPOUND.iloc[0]

-0.0524

In [90]:
one_line_about_doc = nlp(txt)

In [96]:
tt = [t.text for t in one_line_about_doc]
tt.index("AMD")

21

In [91]:
displacy.render(one_line_about_doc, style="dep")

In [97]:
# Extract children of "developer"
print([token.text for token in one_line_about_doc[21].children])

['$']


In [105]:
# Extract previous neighboring node of "developer"
print(one_line_about_doc[21].nbor(-1))

$


In [99]:
# Extract next neighboring node of "developer"
print(one_line_about_doc[21].nbor())

before


In [100]:
# Extract all tokens on the left of "developer"
print([token.text for token in one_line_about_doc[21].lefts])

['$']


In [101]:
# Extract tokens on the right of "developer"
print([token.text for token in one_line_about_doc[21].rights])

[]


In [102]:
# Print subtree of "developer"
print (list(one_line_about_doc[21].subtree))

[$, AMD]


In [88]:
comments.head()

parent_id   created_utc  \
_submission_ id                                 
kok5cp       ghrjlz2  t3_kok5cp  1.609539e+09   
             ghrjnru  t3_kok5cp  1.609539e+09   
kok7my       ghrkap5  t3_kok7my  1.609539e+09   
kokafr       ghrkgqf  t3_kokafr  1.609539e+09   
             ghrkn71  t3_kokafr  1.609539e+09   

                                                                   body  \
_submission_ id                                                           
kok5cp       ghrjlz2  Eat my dongus you fuckin nerd.\n\n*I am a bot,...   
             ghrjnru             i renamed the post , it fits it better   
kok7my       ghrkap5  Hi retards, it's me again with another chart. ...   
kokafr       ghrkgqf  Eat my dongus you fuckin nerd.\n\n*I am a bot,...   
             ghrkn71                 This is too gay, even for WSB. BAN   

                      score  \
_submission_ id               
kok5cp       ghrjlz2      1   
             ghrjnru     -1   
kok7my       ghrkap5     37   
kokafr       ghrkgqf      8   
             ghrkn71      3   

                                                              permalink  \
_submission_ id                                                           
kok5cp       ghrjlz2  /r/wallstreetbets/comments/kok5cp/i_sexually_i...   
             ghrjnru  /r/wallstreetbets/comments/kok5cp/i_sexually_i...   
kok7my       ghrkap5  /r/wallstreetbets/comments/kok7my/ryan_cohengm...   
kokafr       ghrkgqf  /r/wallstreetbets/comments/kokafr/i_sexually_i...   
             ghrkn71  /r/wallstreetbets/comments/kokafr/i_sexually_i...   

                     submission_id subreddit_id  is_root           timestamp  \
_submission_ id                                                                
kok5cp       ghrjlz2     t3_kok5cp     t5_2th52     True 2021-01-01 23:02:28   
             ghrjnru     t3_kok5cp     t5_2th52     True 2021-01-01 23:02:55   
kok7my       ghrkap5     t3_kok7my     t5_2th52     True 2021-01-01 23:08:32   
kokafr       ghrkgqf     t3_kokafr     t5_2th52     True 2021-01-01 23:09:58   
             ghrkn71     t3_kokafr     t5_2th52     True 2021-01-01 23:11:30   

                     PARENT_TYPE PARENT_ID  POLARITY_SCORE_NEG  \
_submission_ id                                                  
kok5cp       ghrjlz2          t3    kok5cp               0.000   
             ghrjnru          t3    kok5cp               0.000   
kok7my       ghrkap5          t3    kok7my               0.000   
kokafr       ghrkgqf          t3    kokafr               0.000   
             ghrkn71          t3    kokafr               0.382   

                      POLARITY_SCORE_NEUT  POLARITY_SCORE_POS  \
_submission_ id                                                 
kok5cp       ghrjlz2                0.924               0.076   
             ghrjnru                0.734               0.266   
kok7my       ghrkap5                1.000               0.000   
kokafr       ghrkgqf                0.924               0.076   
             ghrkn71                0.618               0.000   

                      POLARITY_SCORE_COMPOUND  \
_submission_ id                                 
kok5cp       ghrjlz2                   0.3182   
             ghrjnru                   0.4404   
kok7my       ghrkap5                   0.0000   
kokafr       ghrkgqf                   0.3182   
             ghrkn71                  -0.6523   

                                                                    DOC  \
_submission_ id                                                           
kok5cp       ghrjlz2  (Eat, my, dongus, you, fuckin, nerd.\n\n*I, am...   
             ghrjnru   (i, renamed, the, post, ,, it, fits, it, better)   
kok7my       ghrkap5  (Hi, retards, ,, it, 's, me, again, with, anot...   
kokafr       ghrkgqf  (Eat, my, dongus, you, fuckin, nerd.\n\n*I, am...   
             ghrkn71    (This, is, too, gay, ,, even, for, WSB, ., BAN)   

                                                          

In [ ]:
# i = tasso mensile inoculazione rispetto a popolazione totale (60). Ribaso su tempo trascorso in mesi (3)
(3.8/60)/3

# a = anni necessari per inoculare la popolazione inoculando al tasso i
1/(3.8/60/3)/12

# tasso i mensile a cui inoculare per terminare entro 24 mesi
1/12

# ritardo accumulato
(1/12 - (3.8/60)/3)*3, (3.8/60), (1/12 - (3.8/60)/3)*3 + (3.8/60)